# RSNA Knee — Weak-Label Rule Audit

**Purpose:** measure how well our existing report-derived weak-label rules agree with the expert labels in the competition's fully labelled training studies.

This is an original diagnostic experiment for our project. It reuses the rule-based labeler already present in our weak-supervision baseline; it does not copy the S6 model, train a model, read test reports, or change any submitted model.

## Questions this audit answers

- For each finding, how often does the rule emit a label (coverage)?
- Among the cases it labels, how often does that label match the expert label (accuracy)?
- What are per-target precision, recall, specificity and F1 on the rule-covered subset?
- Which target rules may be too conservative or too noisy?

**Interpretation warning:** the fully labelled subset is only about 58 studies. These are exploratory diagnostics, not a definitive estimate of test performance. Accuracy is computed only where a rule emits a label, so it must always be read alongside coverage. Do not tune many rules to this same small set and then treat its metric as an unbiased final validation result.

No test-side rows or reports are loaded in this notebook. Internet is not required.

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

TARGETS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

def find_competition_root():
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]
    for parent in (Path("/kaggle/input/competitions"), Path("/kaggle/input")):
        if parent.is_dir():
            try:
                candidates.extend([p for p in parent.iterdir() if p.is_dir()])
            except OSError:
                pass
    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except OSError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "train.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the attached competition train.csv. "
        "Attach RSNA Knee Abnormality Detection data to this notebook."
    )

DATA_ROOT = find_competition_root()
TRAIN_CSV = DATA_ROOT / "train.csv"
train_df = pd.read_csv(TRAIN_CSV)

missing_targets = [target for target in TARGETS if target not in train_df.columns]
if missing_targets:
    raise ValueError(f"train.csv missing target columns: {missing_targets}")
if "Report" not in train_df.columns:
    raise ValueError("train.csv has no Report column.")
if "StudyInstanceUID" not in train_df.columns:
    raise ValueError("train.csv has no StudyInstanceUID column.")
if train_df["StudyInstanceUID"].astype(str).duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID values found in train.csv.")

gold_df = train_df.dropna(subset=TARGETS).copy()
if gold_df.empty:
    raise ValueError("No fully expert-labelled studies found.")
gold_values = gold_df[TARGETS].apply(pd.to_numeric, errors="raise").to_numpy(dtype=float)
if not np.isin(gold_values, [0.0, 1.0]).all():
    raise ValueError("Expert target columns must contain only binary 0/1 labels.")

reports = gold_df["Report"].fillna("").astype(str).tolist()
print("Competition input:", DATA_ROOT)
print("Training studies:", len(train_df))
print("Fully expert-labelled studies:", len(gold_df))
print("Expert-labelled studies with non-empty reports:", sum(bool(t.strip()) for t in reports))
print("Note: this audit reads only train.csv; it does not read test.csv or test reports.")

## Reuse the project's current report labeler

The next cell contains the current weak-label rules from our own `kaggle/rsna_knee_submission.ipynb`, kept unchanged for a fair diagnostic comparison. It includes multilingual finding terms and local negation handling. The audit evaluates the existing rules; it does not silently rewrite them based on the outcome.

In [ ]:
def normalize_report(text):
    text = unicodedata.normalize("NFKD", str(text or ""))
    text = text.encode("ascii", "ignore").decode("ascii").lower()
    # Preserve sentence boundaries so negation cannot bleed into another finding.
    text = re.sub(r"[;:\n\r]+", " . ", text)
    text = re.sub(r"[^a-z0-9.]+", " ", text)
    text = re.sub(r"\.+", " . ", text)
    return re.sub(r"\s+", " ", text).strip()
RULES = {
    "ACL": {
        "anchor": r"\b(?:acl|anterior cruciate ligament|ligamento cruzado anterior|vorderes kreuzband|voorste kruisband|ligament croise anterieur)\b",
        "requires_damage": True,
    },
    "MCL": {
        "anchor": r"\b(?:mcl|medial collateral ligament|medial collateral|ligamento colateral medial|mediales kollateralband|innenband|ligament collat(?:eral)? medial)\b",
        "requires_damage": True,
    },
    "Medial Meniscus": {
        "anchor": r"\b(?:medial meniscus|medial meniscal|internal meniscus|menisco medial|menisco interno|menisque medial|menisque interne|innenmeniskus|medial meniskus)\b",
        "requires_damage": True,
    },
    "Lateral Meniscus": {
        "anchor": r"\b(?:lateral meniscus|lateral meniscal|external meniscus|menisco lateral|menisco externo|menisque lateral|ausserer meniskus|außenmeniskus|lateral meniskus)\b",
        "requires_damage": True,
    },
    "Medial OA": {
        "anchor": r"(?:\b(?:medial|inner|internal|mediale|medialen|compartimento medial|compartimento interno)[a-z0-9 ]{0,65}(?:osteoarthritis|arthrosis|degenerative changes|joint space narrowing|cartilage loss|chondrosis|artrosis|gonarthrosis|osteophytes?)\b|\b(?:osteoarthritis|arthrosis|degenerative changes|joint space narrowing|cartilage loss|chondrosis|artrosis|gonarthrosis|osteophytes?)[a-z0-9 ]{0,65}(?:medial|inner|internal|mediale|medialen|compartimento medial|compartimento interno)\b)",
        "requires_damage": False,
    },
    "Lateral OA": {
        "anchor": r"(?:\b(?:lateral|outer|laterale|lateralen|compartimento lateral|compartimento externo)[a-z0-9 ]{0,65}(?:osteoarthritis|arthrosis|degenerative changes|joint space narrowing|cartilage loss|chondrosis|artrosis|gonarthrosis|osteophytes?)\b|\b(?:osteoarthritis|arthrosis|degenerative changes|joint space narrowing|cartilage loss|chondrosis|artrosis|gonarthrosis|osteophytes?)[a-z0-9 ]{0,65}(?:lateral|outer|laterale|lateralen|compartimento lateral|compartimento externo)\b)",
        "requires_damage": False,
    },
    "PF OA": {
        "anchor": r"(?:\b(?:patellofemoral|femoropatellar|retropatellar|patellar cartilage|trochlear cartilage)[a-z0-9 ]{0,65}(?:osteoarthritis|arthrosis|degenerative changes|cartilage loss|chondrosis|artrosis|osteophytes?|chondral defects?)\b|\b(?:osteoarthritis|arthrosis|degenerative changes|cartilage loss|chondrosis|artrosis|osteophytes?|chondral defects?)[a-z0-9 ]{0,65}(?:patellofemoral|femoropatellar|retropatellar|patellar cartilage|trochlear cartilage)\b)",
        "requires_damage": False,
    },
    "Effusion": {
        "anchor": r"\b(?:effusion|joint fluid|joint effusion|hydrops|derrame articular|derrame|gelenkerguss|erguss|articular fluid)\b",
        "requires_damage": False,
    },
    "Synovitis": {
        "anchor": r"\b(?:synovitis|sinovitis|synovial hypertrophy|synovial thickening|synovial proliferation|synoviale hyperplasie)\b",
        "requires_damage": False,
    },
    "Baker's": {
        "anchor": r"\b(?:baker.?s cyst|popliteal cyst|quiste popliteo|quiste de baker|baker zyste|bakerzyste|poplitealzyste)\b",
        "requires_damage": False,
    },
    "Contusion": {
        "anchor": r"\b(?:bone contusion|bone bruise|marrow edema|bone marrow edema|bone marrow oedema|edema oseo|edema medular|contusion ossea|knochenmarkodem|trabecular injury)\b",
        "requires_damage": False,
    },
    "Fracture": {
        "anchor": r"\b(?:fracture|fractured|fractura|fraktur|fratura|stress fracture|insufficiency fracture|fracture line)\b",
        "requires_damage": False,
    },
}

DAMAGE_WORDS = re.compile(
    r"\b(?:tear|tears|torn|rupture|ruptured|ruptur|ruptura|rotura|"
    r"desgarro|dechirure|riss|ruptuur|scheur|sprain|injury|injured|"
    r"avulsion|discontinuity|partial tear|complete tear|bucket handle|"
    r"radial tear|root tear|mucoid degeneration)\b"
)
NEGATION_PREFIX = re.compile(
    r"(?:\bno\b|\bwithout\b|\bnot\b|\bnegative for\b|"
    r"\babsence of\b|\bno evidence of\b|\bno signs of\b|"
    r"\bsin\b|\bsin signos de\b|\bsin evidencia de\b|"
    r"\bno se observa\b|\bno se identifican\b|\bningun\b|"
    r"\bkein\b|\bkeine\b|\bkeinen\b|\bkeiner\b|\bohne\b|"
    r"\bnicht\b|\bgeen\b|\bgeen aanwijzing voor\b|"
    r"\bpas de\b|\bsans\b|\babsence de\b|\bintact\b|\bnormal\b|\bunremarkable\b|\bpreserved\b|\bwithin normal limits\b)"
)
NEGATION_SUFFIX = re.compile(
    r"\b(?:absent|not seen|not identified|not visualized|not present|"
    r"without evidence|intact|normal|unremarkable|preserved|within normal limits|"
    r"no tear|no rupture|sin rotura|sin desgarro|integro|conservado|"
    r"unauffallig|intakt|kein hinweis)\b"
)

def weak_labels_from_report(report):
    text = normalize_report(report)
    values = np.zeros(len(TARGETS), dtype=np.float32)
    mask = np.zeros(len(TARGETS), dtype=np.float32)

    for target_index, target in enumerate(TARGETS):
        rule = RULES[target]
        for match in re.finditer(rule["anchor"], text):
            # Restrict negation and injury evidence to the current sentence.
            sentence_start = text.rfind(".", 0, match.start()) + 1
            sentence_end = text.find(".", match.end())
            if sentence_end < 0:
                sentence_end = len(text)
            left = text[max(sentence_start, match.start() - 90):match.start()]
            right = text[match.end():min(sentence_end, match.end() + 75)]
            context = left[-80:] + " " + text[match.start():match.end()] + " " + right[:80]

            # A negation must be close to the target phrase, not merely elsewhere
            # in the same report. Up to six intervening tokens covers common
            # constructions such as "no evidence of a tear of the ACL".
            negated_prefix = re.search(
                NEGATION_PREFIX.pattern + r"(?:\s+[a-z0-9]+){0,6}\s*$",
                left,
            ) is not None
            negated = (
                negated_prefix
                or NEGATION_PREFIX.search(match.group(0)) is not None
                or NEGATION_SUFFIX.search(right) is not None
            )
            if rule["requires_damage"]:
                has_damage = DAMAGE_WORDS.search(context) is not None
                if negated:
                    values[target_index] = 0.0
                    mask[target_index] = 1.0
                    break
                if has_damage:
                    values[target_index] = 1.0
                    mask[target_index] = 1.0
                    break
                # Merely mentioning a ligament or meniscus is not a positive.
                continue

            values[target_index] = 0.0 if negated else 1.0
            mask[target_index] = 1.0
            break

    return values, mask

In [ ]:
def safe_rate(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan

def binary_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    if y_true.shape != y_pred.shape or y_true.ndim != 1:
        raise ValueError("Expected equally sized one-dimensional arrays.")
    if not np.isin(y_true, [0, 1]).all() or not np.isin(y_pred, [0, 1]).all():
        raise ValueError("Metrics require binary 0/1 values.")
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = safe_rate(tp, tp + fp)
    recall = safe_rate(tp, tp + fn)
    specificity = safe_rate(tn, tn + fp)
    f1 = safe_rate(2 * tp, 2 * tp + fp + fn)
    return {
        "n_covered": int(y_true.size),
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "accuracy": safe_rate(tp + tn, y_true.size),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
    }

# The parser in the preceding cell is the same labeler used by the competition
# baseline. We evaluate only fully expert-labelled training studies.
rule_values = []
rule_masks = []
for report in reports:
    values, mask = weak_labels_from_report(report)
    rule_values.append(values)
    rule_masks.append(mask)

rule_values = np.asarray(rule_values, dtype=np.float32)
rule_masks = np.asarray(rule_masks, dtype=np.float32)
if rule_values.shape != gold_values.shape or rule_masks.shape != gold_values.shape:
    raise ValueError("Weak-label output dimensions do not match the expert labels.")

rows = []
for j, target in enumerate(TARGETS):
    covered = rule_masks[:, j] > 0
    target_metrics = binary_metrics(gold_values[covered, j], rule_values[covered, j].astype(np.int64))
    rows.append({
        "target": target,
        "gold_studies": int(len(gold_df)),
        "nonempty_reports": int(sum(bool(t.strip()) for t in reports)),
        "covered_studies": int(covered.sum()),
        "coverage_pct": safe_rate(int(covered.sum()), len(gold_df)) * 100.0,
        "gold_positive_in_covered": int(gold_values[covered, j].sum()),
        "rule_positive_in_covered": int(rule_values[covered, j].sum()),
        **target_metrics,
    })

audit_df = pd.DataFrame(rows)
print("\nWeak-label rule audit by target:")
print(audit_df.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
covered_pairs = int(rule_masks.sum())
correct_pairs = int(((rule_values == gold_values) * (rule_masks > 0)).sum())
total_pairs = int(gold_values.size)
overall_covered_accuracy = safe_rate(correct_pairs, covered_pairs)
overall_coverage = safe_rate(covered_pairs, total_pairs)
print(f"\nCovered target-study pairs: {covered_pairs}/{total_pairs} ({overall_coverage:.1%})")
print(f"Accuracy among rule-covered pairs: {overall_covered_accuracy:.3f}" if np.isfinite(overall_covered_accuracy) else "Accuracy among rule-covered pairs: undefined")

out_path = Path("/kaggle/working/weak_label_audit.csv")
audit_df.to_csv(out_path, index=False, na_rep="")
print("Saved:", out_path, f"({out_path.stat().st_size:,} bytes)")

# Independent sanity test of the metrics implementation; 2 of 4 predictions are correct.
example = binary_metrics([1, 1, 0, 0], [1, 0, 0, 1])
assert example["accuracy"] == 0.5 and example["precision"] == 0.5 and example["recall"] == 0.5
assert int(audit_df["covered_studies"].max()) <= len(gold_df)
print("Metric sanity tests passed.")

## How to use the result

Open `weak_label_audit.csv` in the Output panel and inspect coverage before interpreting accuracy. A rule with high accuracy but very low coverage may help only a small fraction of training studies. A rule with low precision can inject harmful pseudo-labels. Because there are few expert-labelled studies, any rule revisions should be treated as hypotheses and compared with a frozen validation protocol.

This audit is a research artifact, not a replacement for the selected 0.943 competition entry. The actual competition metric is macro ROC AUC from the private hidden-test rerun.